In [1]:
import gc
import warnings
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

In [5]:
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

In [6]:
def find_best_threshold(y_true, probabilities):

    best_threshold = 0.5
    best_score = 0.0

    # Test candidate thresholds from 0.10 to 0.80
    for threshold in np.arange(0.10, 0.80, 0.02):

        predictions = (probabilities >= threshold).astype(int)

        score = fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0
        )

        if score > best_score:
            best_score = score
            best_threshold = round(float(threshold), 2)

    return best_threshold

In [7]:
# Load the dataset
data_path = "diabetic_data.csv"

df = pd.read_csv(data_path)

print("Dataset Loaded Successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset Loaded Successfully!
Rows: 101766
Columns: 50


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [8]:
# Replace '?' with NaN
df.replace("?", np.nan, inplace=True)

# Display missing values
print("Missing Values in Dataset:")

print(
    df.isnull().sum().sort_values(ascending=False).head(10)
)

Missing Values in Dataset:
weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
patient_nbr              0
dtype: int64


In [9]:
# Keep first encounter per patient
df = df.drop_duplicates(
    subset=["patient_nbr"],
    keep="first"
)

# Remove any remaining duplicate rows
df = df.drop_duplicates()

print("Rows after removing duplicate patient records:", len(df))

Rows after removing duplicate patient records: 71518


In [10]:
# Target mapping
# <30 = 1 (Readmitted within 30 days)
# NO  = 0 (Not readmitted)
# >30 = 0 (Readmitted after 30 days)

target_mapping = {
    "<30": 1,
    "NO": 0,
    ">30": 0
}

df["target"] = df["readmitted"].map(target_mapping).astype(int)

print("Target variable created successfully!")

Target variable created successfully!


In [11]:
# Count positive and negative classes
pos_count = (df["target"] == 1).sum()
neg_count = (df["target"] == 0).sum()

pos_pct = (pos_count / len(df)) * 100

print("Target Class Distribution:")

print(
    f"Readmitted within 30 days (1): {pos_count} ({pos_pct:.2f}%)"
)

print(
    f"Not readmitted within 30 days (0): {neg_count} ({100 - pos_pct:.2f}%)"
)

Target Class Distribution:
Readmitted within 30 days (1): 6293 (8.80%)
Not readmitted within 30 days (0): 65225 (91.20%)


In [12]:
# Columns that should not be used as model features
drop_columns = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "target"
]

# Identify zero-variance columns
zero_variance_cols = [
    c for c in df.columns
    if df[c].nunique(dropna=False) <= 1
]

if zero_variance_cols:
    print("Removing zero-variance columns:", zero_variance_cols)

    drop_columns.extend(zero_variance_cols)

# Separate features and target
X = df.drop(
    columns=drop_columns,
    errors="ignore"
).copy()

y = df["target"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Removing zero-variance columns: ['examide', 'citoglipton', 'glimepiride-pioglitazone']
Features shape: (71518, 44)
Target shape: (71518,)


In [13]:
# Convert nominal ID columns into strings
nominal_id_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in nominal_id_cols:

    if col in X.columns:
        X[col] = X[col].astype(str)

print("Categorical ID conversion completed!")

Categorical ID conversion completed!


In [14]:
# Identify numerical columns
numerical_columns = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical columns
categorical_columns = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Store feature names in original order
feature_columns = list(X.columns)

print("Total Features:", len(feature_columns))

print("\nNumerical Features:", len(numerical_columns))
print(numerical_columns)

print("\nCategorical Features:", len(categorical_columns))
print(categorical_columns)

Total Features: 44

Numerical Features: 8
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical Features: 36
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [15]:
# Split dataset into 80% training and 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

Training Samples: 57214
Testing Samples: 14304


In [16]:
# Numerical Transformer
# 1. Fill missing values using median
# 2. Standardize numerical features

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical preprocessing configured!")

Numerical preprocessing configured!


In [18]:
# Categorical Transformer
# 1. Fill missing values using constant value
# 2. Convert categorical values into numerical format

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical preprocessing configured!")

Categorical preprocessing configured!


In [19]:
# Combine numerical and categorical transformations

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ],
    remainder="drop"
)

print("Complete preprocessing pipeline configured!")

Complete preprocessing pipeline configured!


In [20]:
print("Tuning Decision Tree Classifier with GridSearchCV...")

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

dt_param_grid = {
    "classifier__max_depth": [5, 10, 15],
    "classifier__min_samples_split": [10, 20]
}

dt_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    cv=3,
    scoring="average_precision",
    n_jobs=1,
    verbose=0
)

dt_search.fit(X_train, y_train)

best_dt = dt_search.best_estimator_

print("Decision Tree Best Parameters:")
print(dt_search.best_params_)

Tuning Decision Tree Classifier with GridSearchCV...
Decision Tree Best Parameters:
{'classifier__max_depth': 10, 'classifier__min_samples_split': 20}


In [21]:
print("Tuning Logistic Regression with GridSearchCV...")

lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=500,
        solver="liblinear",
        random_state=42
    ))
])

lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1.0]
}

lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=3,
    scoring="average_precision",
    n_jobs=1,
    verbose=0
)

lr_search.fit(X_train, y_train)

best_lr = lr_search.best_estimator_

print("Logistic Regression Best Parameters:")
print(lr_search.best_params_)

Tuning Logistic Regression with GridSearchCV...
Logistic Regression Best Parameters:
{'classifier__C': 0.1}


In [22]:
print("Tuning Random Forest with RandomizedSearchCV...")

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

rf_param_dist = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [8, 12, 16],
    "classifier__min_samples_split": [5, 10]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=4,
    cv=3,
    scoring="average_precision",
    random_state=42,
    n_jobs=1,
    verbose=0
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("Random Forest Best Parameters:")
print(rf_search.best_params_)

Tuning Random Forest with RandomizedSearchCV...
Random Forest Best Parameters:
{'classifier__n_estimators': 100, 'classifier__min_samples_split': 5, 'classifier__max_depth': 16}


In [23]:
candidate_models = {
    "Decision Tree": best_dt,
    "Logistic Regression": best_lr,
    "Random Forest": best_rf
}

comparison_records = []
evaluated_models = {}

print("All three trained models stored successfully!")

All three trained models stored successfully!


In [24]:
print("=" * 70)
print("MODEL EVALUATION AND THRESHOLD OPTIMIZATION")
print("=" * 70)

for name, model in candidate_models.items():

    print("\n" + "-" * 50)
    print("EVALUATION:", name)
    print("-" * 50)

    # Generate training cross-validation probabilities
    train_cv_probs = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=3,
        method="predict_proba"
    )[:, 1]

    # Find best threshold using F2 score
    tuned_threshold = find_best_threshold(
        y_train,
        train_cv_probs
    )

    print("Tuned Threshold:", tuned_threshold)

MODEL EVALUATION AND THRESHOLD OPTIMIZATION

--------------------------------------------------
EVALUATION: Decision Tree
--------------------------------------------------
Tuned Threshold: 0.44

--------------------------------------------------
EVALUATION: Logistic Regression
--------------------------------------------------
Tuned Threshold: 0.42

--------------------------------------------------
EVALUATION: Random Forest
--------------------------------------------------
Tuned Threshold: 0.46


In [25]:
    # Predict probabilities on test data
    test_probs = model.predict_proba(X_test)[:, 1]

    # Apply tuned threshold
    test_preds = (
        test_probs >= tuned_threshold
    ).astype(int)

    print("Test predictions generated!")

Test predictions generated!


In [26]:
    # Calculate evaluation metrics

    acc = accuracy_score(y_test, test_preds)

    prec = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f2 = fbeta_score(
        y_test,
        test_preds,
        beta=2,
        zero_division=0
    )

    roc_auc = roc_auc_score(y_test, test_probs)

    pr_auc = average_precision_score(y_test, test_probs)

    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)
    print("F2 Score :", f2)
    print("ROC-AUC  :", roc_auc)
    print("PR-AUC   :", pr_auc)

Accuracy : 0.4450503355704698
Precision: 0.11477679086399815
Recall   : 0.7903097696584591
F1 Score : 0.20044319097502014
F2 Score : 0.36300620211601603
ROC-AUC  : 0.6695183258537761
PR-AUC   : 0.18374420433708988


In [27]:
    comparison_records.append({
        "Model": name,
        "Threshold": tuned_threshold,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "F2 Score": round(f2, 4),
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC": round(pr_auc, 4)
    })

    evaluated_models[name] = {
        "model": model,
        "threshold": tuned_threshold,
        "metrics": {
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "F2 Score": f2,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc
        }
    }

    print("Evaluation results stored!")

Evaluation results stored!


In [28]:
    # Generate confusion matrix
    cm = confusion_matrix(y_test, test_preds)

    print("\nConfusion Matrix:")
    print(cm)

    print(f"TN: {cm[0, 0]}")
    print(f"FP: {cm[0, 1]}")
    print(f"FN: {cm[1, 0]}")
    print(f"TP: {cm[1, 1]}")


Confusion Matrix:
[[5371 7674]
 [ 264  995]]
TN: 5371
FP: 7674
FN: 264
TP: 995


In [29]:
    print("\nClassification Report:")

    print(classification_report(
        y_test,
        test_preds,
        target_names=[
            "No Readmission (0)",
            "Readmitted <30d (1)"
        ],
        digits=4,
        zero_division=0
    ))


Classification Report:
                     precision    recall  f1-score   support

 No Readmission (0)     0.9531    0.4117    0.5751     13045
Readmitted <30d (1)     0.1148    0.7903    0.2004      1259

           accuracy                         0.4451     14304
          macro avg     0.5340    0.6010    0.3877     14304
       weighted avg     0.8794    0.4451    0.5421     14304



In [30]:
comparison_df = pd.DataFrame(comparison_records)

print("=" * 70)
print("MODEL COMPARISON TABLE")
print("=" * 70)

print(comparison_df.to_string(index=False))

MODEL COMPARISON TABLE
        Model  Threshold  Accuracy  Precision  Recall  F1 Score  F2 Score  ROC-AUC  PR-AUC
Random Forest       0.46    0.4451     0.1148  0.7903    0.2004     0.363   0.6695  0.1837


In [31]:
# Select model based on F2 Score and PR-AUC

best_row = comparison_df.sort_values(
    by=["F2 Score", "PR-AUC"],
    ascending=False
).iloc[0]

best_model_name = best_row["Model"]

best_package = evaluated_models[best_model_name]

print("Best Selected Model:", best_model_name)

print("F2 Score:", best_row["F2 Score"])
print("PR-AUC:", best_row["PR-AUC"])

print("Selected Threshold:", best_package["threshold"])

Best Selected Model: Random Forest
F2 Score: 0.363
PR-AUC: 0.1837
Selected Threshold: 0.46


In [32]:
# Package complete model information

saved_payload = {
    "model": best_package["model"],
    "threshold": best_package["threshold"],
    "feature_columns": feature_columns,
    "target_mapping": target_mapping
}

print("Model package prepared successfully!")

Model package prepared successfully!


In [33]:
# Save model using Joblib

output_filename = "hospital_readmission_model.pkl"

joblib.dump(
    saved_payload,
    output_filename
)

print("Model saved successfully!")
print("Filename:", output_filename)

Model saved successfully!
Filename: hospital_readmission_model.pkl


In [34]:
print("Saved Model Contents:")

print("Model:", best_model_name)

print("Threshold:", saved_payload["threshold"])

print(
    "Number of Features:",
    len(saved_payload["feature_columns"])
)

print("Target Mapping:", saved_payload["target_mapping"])

Saved Model Contents:
Model: Random Forest
Threshold: 0.46
Number of Features: 44
Target Mapping: {'<30': 1, 'NO': 0, '>30': 0}


In [35]:
print("=" * 70)
print("FINAL BEST MODEL TEST METRICS SUMMARY")
print("=" * 70)

print("Best Model:", best_model_name)

print("Threshold:", best_package["threshold"])

for metric_name, metric_val in best_package["metrics"].items():

    print(f"{metric_name:<11}: {metric_val:.4f}")

print("=" * 70)

FINAL BEST MODEL TEST METRICS SUMMARY
Best Model: Random Forest
Threshold: 0.46
Accuracy   : 0.4451
Precision  : 0.1148
Recall     : 0.7903
F1 Score   : 0.2004
F2 Score   : 0.3630
ROC-AUC    : 0.6695
PR-AUC     : 0.1837


In [36]:
import os

print(
    "Model file exists:",
    os.path.exists("hospital_readmission_model.pkl")
)

if os.path.exists("hospital_readmission_model.pkl"):

    file_size = os.path.getsize(
        "hospital_readmission_model.pkl"
    ) / 1024

    print(f"Model file size: {file_size:.2f} KB")

# Clean up memory
gc.collect()

print("\nMODEL TRAINING AND SAVING COMPLETED SUCCESSFULLY!")

Model file exists: True
Model file size: 7181.54 KB

MODEL TRAINING AND SAVING COMPLETED SUCCESSFULLY!
